<a href="https://colab.research.google.com/github/alscop/ESAA-26-1/blob/main/ESAA_OB_week06_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 딥러닝 파이토치 교과서 4장 p.138-163

In [2]:
import torch

# 4장. 딥러닝 시작

## 4.1 인공 신경망의 한계와 답러닝 출현

퍼셉트론 구조 - 신경망(딥러닝)의 기원이 되는 알고리즘.

다수의 신호를 입력으로 하나의 신호 출력.  
흐른다/안 흐른다(1/0) 정보를 앞으로 전달. (0일 경우 무시)


### AND 게이트

모든 입력이 '1'일 때 작동.  
입력 중 어떤 하나라도 '0'이라면 작동 멈춤

### OR 게이트
입력에서 1개 이상 '1'일 때 작동.  
입력 모두가 '0'인 경우 제외하고 '1'값을 출력

### XOR 게이트
배타적 논리합.  
입력 두 개 중 한개만 '1'일 때 작동  
-> 비선형적으로 분리되어 제대로 된 분류 어려움.

단층 퍼셉트론: AND, OR 연산 학습 가능, XOR 연산 학습 불가능.
-> 다층 퍼셉트론: 입력층-출력층 사이에 하나 이상의 중간층(은닉층) 둠. 비선형적으로 분리되는 데이터에서도 학습 가능!

이때 은닉층이 여러 개라면 '심층 신경망(DNN)', '딥러닝'


## 4.2 딥러닝 구조

### 4.2.1. 딥러닝 용어

- 층
  - 입력층
  - 은닉층
  - 출력층

- 가중치: 노드와 노드 간 연결 강도 조정
- 바이어스: 가중치에 더해주는 상수.   
(하나의 뉴런에서 활성화 함수를 거쳐 최종적으로 출력되는 값을 조절하는 역할)

- 가중합(= 전달 함수) : '가중치*신호+바이어스'의 합계.  
이 가중합을 활성화 함수로 보내기 때문에 전달함수라고도 함.

- 함수
  - 활성화 함수: 신호 입력받아 이를 적절히 처리, 출력해주는 함수(비선형)  
    - 시그모이드: 0~1 사이에서 비선형 형태로 변형해줌. 분류. 딥러닝에서는 기울기 소멸 문제로 사용 잘 안 함.
    - 하이퍼볼릭 탄젠트: -1~1 사이에서 비선형 형태로 변형해줌. 시그모이드의 결괏값 평균이 0이 아니라 양수로 편향되는 문제 해결, 기울기 소멸 문제는 여전함.
    - 렐루 함수: 입력값이 음수면 0 출력, 양수면 x 출력. 경사하강법에 영향 없어 학습 속도 빠름. 기울기 소멸 문제 발생X. BUT 학습능력 감소
    - 리키 렐루 함수: 렐루 함수 문제 해결. 입력값이 음수면 0.001처럼 매우 작은 수 반환. 입력값 수렴 구간 제거됨.
    - 소프트맥스 함수: 입력값을 0~1 사이에 출력되도록 정규화. 출력 값들의 총합이 항상 1이 되도록 함.

  - 손실 함수: 가중치 학습(경사하강법)을 위해 출력 함수의 결과와 실제 값 간의 오차 측정하는 함수.
    - 평균 제곱 오차(MSE): 실제값과 예측값의 차이를 제곱해 평균 낸 것. 값이 작을수록 예측력 good. 회귀.
    - 크로스 엔트로피 오차(CEE): 분류에서 원-핫 인코딩 사용시 사용 가능. 두 개의 확률 분포 차이 이용, 시그모이드 영향 덜 받음.  
    MSE 손실함수+시그모이드 활성화 함수 결합 시 기울기 매끄럽지 못하고 학습 속도 매우 느린 문제 발생(시그모이드 때문) -> CEE는 학습 속도 빠름.  
    신경망 출력(추정값)의 로그값 * 정답 레이블의 합계

    






> 파이토치에서 렐루 함수, 소프트맥스 함수 구현

In [3]:
class Net(torch.nn.Module):
  def __init__(self, n_feature, n_hidden, n_output):
    super(Net, self).__init__()
    self.hidden = torch.nn.Linear(n_feature, n_hidden) # 은닉층
    self.relu = torch.nn.ReLu(inplace=True)
    self.out = torch.nn.Linear(n_hidden, n_output) # 출력층
    self.softmax = torch.nn.Softmax(dim=n_output)
  def forward(self, x):
    x = self.hidden(x)
    x = self.relu(x) # 은닉층을 위한 렐루 활성화 함수
    x = self.out(x)
    x = self.softmax(x) # 출력층을 위한 소프트맥스 활성화 함수
    return x

> 파이토치에서 평균 제곱 오차 구하기

In [4]:
'''
import torch

loss_fn = torch.nn.MSELoss(reduction='sum')
y_pred = model(x)
loss = loss_fn(y_pred, y)
'''

"\nimport torch\n\nloss_fn = torch.nn.MSELoss(reduction='sum')\ny_pred = model(x)\nloss = loss_fn(y_pred, y)\n"

> 파이토치에서 크로스 엔트로피 오차 구하기

In [6]:
'''
loss = nn.CrossEntropyLoss()
input = torch.randn(5, 6, requires_grad=True) # torch.randn은 평균이 0, 표준편차가 1인 가우시안 정규 분포  이용해 숫자 생성
target = torch.empty(3, dtype=torch.long).random_(5) # torch.empty 는 dtype torch.float32의 랜덤한 값으로 채워진 텐서를 반환
output = loss(input, target)
output.backward()
'''

'\nloss = nn.CrossEntropyLoss()\ninput = torch.randn(5, 6, requires_grad=True) # torch.randn은 평균이 0, 표준편차가 1인 가우시안 정규 분포  이용해 숫자 생성\ntarget = torch.empty(3, dtype=torch.long).random_(5) # torch.empty 는 dtype torch.float32의 랜덤한 값으로 채워진 텐서를 반환\noutput = loss(input, target)\noutput.backward()\n'

### 4.2.2 딥러닝 학습

1. 순전파
네트워크에 훈련 데이터가 들어올 때 발생.  
모든 뉴런이 이전 층의 뉴런에서 수신한 정보에 변환(가중합 및 활성화 함수)을 적용하여 다음 층(은닉층)의 뉴런으로 전송하는 방식.  
예측 값은 최종 층(출력층)에 도달


2. 역전파
손실함수로 손실(오차) 추정. 0이 이상적.  
손실 함수 비용이 0에 가까워지도록 모델 훈련 반복, 가중치 조정.
정보가 역으로 전파되어 역전파라고 부름.  


### 4.2.3 딥러닝의 문제점과 해결 방안

핵심: 활성화 함수가 적용된 여러 은닉층을 결합하여 비선형 영역을 표현  

은닉층 개수가 많을수록 데이터 분류 잘 됨. BUT 3가지 문제점이 ...


#### 1. 과적합 문제

데이터 일부만 사용하는 학습 과정에서는 오차 감소, 검증 데이터에 대해서는 오차 증가.  
-> 드롭아웃(dropout)  
: 학습 과정 중 임의로 일부 노드들을 학습에서 제외시킴






> 파이토치에서 드롭아웃 구현

In [7]:
class DropoutModel(torch.nn.Module):
  def __init__(self):
    super(DropoutModel, self).__init__()
    self.layer1 = torch.nn.Linear(784, 1200)
    self.dropout1 = torch.nn.Dropout(0.5) # 50% 노드 무작위로 선택해 사용하지 않겠다는 뜻
    self.layer2 = torch.nn.Linear(1200, 1200)
    self.dropout2 = torch.nn.Dropout(0.5)
    self.layer3 = torch.nn.Linear(1200, 10)

  def forward(self, x):
    x = F.relu(self.layer1(x))
    x = self.dropout1(x)
    x = F.relu(self.layer2(x))
    x = self.dropout2(x)
    return self.layer3(x)

#### 2. 기울기 소멸 문제

은닉층 많은 신경망에서 주로 발생  
출력층에서 은닉층으로 전달되는 오차가 크게 줄어들어(기울기 소멸) 학습량이 0에 가까워져 학습 더디고, 오차 못 줄인 채 수렴되는 현상.  

-> 시그모이드, 하이퍼볼릭 탄젠트 대신 '렐루 활성화 함수' 사용하면 해결 가능.

#### 3. 성능이 나빠지는 문제

손실 함수 비용 최소 지점 찾을 때까지 기울기 낮은 쪽으로 계속 이동시키는 경사 하강법에서 성능 나빠지기도 함.

-> 확률적 경사 하강법, 미니 배치 경사 하강법 사용하면 해결 가능.

- 배치 경사 하강법: 전체  데이터셋에 대한 오류를 구한 후 기울기를 한 번만 계산하여 모델의 파라미터를 업데이트.   
전체 훈련 데이터셋에 대해 가중치 편미분.
학습속도 느림.

- 확률적 경사 하강법:  임의로 선택한 데이터에 대해 기울기를 계산.  
계산 속도 빠름. 정확도 낮을 수 있음. 파라미터 변경 폭 불안정

- 미니 배치 경사 하강법:  전체 데이터셋을 미니 배치 여러 개로 나누고 미니 배치 한 개마다 기울기를 구한 후 그것의 평균 기울기를 이용하여 모델을 업데이트  
전체 데이터 계산보다 빠르고, 확률적 보다 안정적.  




> 파이토치에서 미니 배치 경사 하강법 구현

In [9]:
'''
class CustomDataset(Dataset):
  def __init__(self):
    self.x_data = [[1,2,3], [4,5,6], [7,8,9]]
    self.y_data = [[12], [18], [11]]
    def __len__(self):
      return len(self.x_data)
    def __getitem__(self, idx):
      x = torch.FloatTensor(self.x_data[idx])
      y = torch.FloatTensor(self.y_data[idx])
      return x, y
dataset = CustomDataset()
dataloader = dataloader(
    dataset, # 데이터셋
    batch_size = 2, # 미니 배치 크기로 2의 제곱수를 사용하겠다는 뜻
    shuffle=True, # 데이터를 불러올 때마다 랜덤으로 섞어서 가져옴
)
'''

'\nclass CustomDataset(Dataset):\n  def __init__(self):\n    self.x_data = [[1,2,3], [4,5,6], [7,8,9]]\n    self.y_data = [[12], [18], [11]]\n    def __len__(self):\n      return len(self.x_data)\n    def __getitem__(self, idx):\n      x = torch.FloatTensor(self.x_data[idx])\n      y = torch.FloatTensor(self.y_data[idx])\n      return x, y\ndataset = CustomDataset()\ndataloader = dataloader(\n    dataset, # 데이터셋\n    batch_size = 2, # 미니 배치 크기로 2의 제곱수를 사용하겠다는 뜻\n    shuffle=True, # 데이터를 불러올 때마다 랜덤으로 섞어서 가져옴\n)\n'

#### Note) 옵티마이저

확률적 경사 하강법의 파라미터 변경 폭이 불안정한 문제 해결 위해 학습 속도, 운동량 조정하는 옵티마이저 적용해볼 수 있음.   

- 속도 조정하는 방법
  - 아다그라드:  변수(가중치)의 업데이트 횟수에 따라 학습률을 조정. 많이 변화 안 하는 변수들의 학습률을 크게함.(최적값에 근접했을 것이라는 가정)  
  `optimizer = torch.optim.Adagrad(model.parameters(), lr=0.01` # 학습률 기본값은 1e-2  
  기울기 0 수렴 문제 있음, 사용 X
  - 아다델타: 아다그라드 G 값 커질때 학습 멈추는 문제 해결. 학습률을 D함수(가중치 변화량 크기 누적값)으로 변환해 학습률에 대한 하이퍼 파라미터 필요 X  
  `optimizer = torch.optim.Adadelta(model.parameters(), lr=1.0)` # 학습률 기본값은 1.0
  - 알엠에스프롭: 아다그라드의 G값이 무한히 커지는 문제 방지. G함수에 감마 추가, 학습률 크기를 비율로 조정 가능.  
  `optimizer = torch.optim.RMSprop(model.parameters(), lr=0.01)` # 학습률 기본값은 1e-2

- 운동량을 조정하는 방법
  - 모멘텀: 경사하강법처럼 매번 기울기 도출 but 가중치 수정 전 이전 수정 방향(+, -)을 참고해 같은 방향으로 일정한 비율만 수정. (+, -) 방향으로 지그재그 수정되는 현상 줄어들고, 관성 효과 얻을 수 있음. SGD(확률적 경사 하강법)와 함께 사용.  
  `optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)`  
  momentum 값은 0.9에서 시작, 0.95, 0.99처럼 조금씩 증가시키며 사용.
  
  - 네스테로프 모멘텀:  모멘텀 값이 적용된 지점에서 기울기 값 계산. 모멘텀으로 절반 정도 이동한 후 어떤 방식으로 이동해야 하는지 다시 계산하여 결정하므로 기존 모멘텀의 관성으로 멈춰야하는데 훨씬 멀리가는 단점 극복 가능. 빠른 이동속도, 적절한 시점에서 제동 걸기 용이.  
  `optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=Ture)` # nesterov 기본값은 False

- 속도와 운동량에 대한 혼용 방법
  - 아담: 모멘텀 + 알엠에스프롭 장점 결합한 경사하강법.  
  `optimizer = torch.optim.Adam(model.parameters(), lr=0.01)` # 학습률 기본값은 1e-3



### 4.2.4 딥러닝을 사용할 때 이점

#### 특성 추출
데이터별로 어떤 특정을 가지고 있는지 찾아내고 그것을 토대로 데이터를 벡터로 변환하는 작업.  

머신러닝은 도메인 지식이 필요했으나 딥러닝은 특성 추출 과정이 알고리즘에 통합됨.

#### 빅데이터의 효율적 활용

딥러닝 학습을 이용한 특성 추출은 데이터 사례가 많을수록 성능이 향상됨

## 4.3 딥러닝 알고리즘

심층 신경망을 사용한다는 공통점 존재. 목적에 따라 분류

### 4.3.1 심층 신경망(DNN)

입력층과 출력층 사이에 다수의 은닉층을 포함하는 인공 신경망
- 다양한 비선형적 관계 학습 가능
- 연산량 많음
- 기울기 소멸 문제

-> 드롭아웃, 렐루함수, 배치 정규화 등으로 해결

### 4.3.2 합성곱 신경망(CNN)

합성곱층과 풀링층을 포함하는 이미지 처리 성능이 좋은 인공 신경망 알고리즘.  
- 영상 및 사진이 포함된 이미지 데이터에서 객체 탐색, 객체 위치 찾아내는데 유용
- 인식을 위한 패턴 찾는데 특히 유용
- 각 층의 입출력 형상을 유지
- 이미지의 공간 정보를 유지하면서 인접 이미지와 차이가 있는 특징을 효과적으로 인식
- 복수 필터로 이미지의 특징을 추출하고 학습
- 추출한 이미지의 특정을 모으고 강화하는 풀링층 존재
- 필터를 공유 파라미터로 시용하기 때문에 일반 인공 신경망과 비교하여 학습 파라미터가 매우 적음

### 4.3.3 순환 신경망(RNN)

시계열 데이터(음악, 영상 등) 같은 시간 흐름에 따라 변화하는 데이터를 학습하기 위함.  
'순환': 자기 자신을 참조, 현재 결과가 이전 결과와 연관이 있다는 의미
- 시간성 가진 데이터 多
- 시간성 정보를 이용하여 데이터의 특징을 잘 다룸
- 시간에 따라 내용이 변하므로 데이터는 동적이고 길이가 가변적
- 매우 긴 데이터를 처리히는 연구가 활발히 진행 중

기울기 소멸 문제 -> 메모리 개념을 도입한 LSTM 많이 사용됨.  
- 자연어 처리 분야와 좋은 궁합.

### 4.3.4. 제한된 볼츠만 머신

가시층과 은닉층으로 구성됨.  
가시층-은닉층만 연결되고, 가시층끼리/은닉층끼리 연결은 없음.

- 차원 감소, 분류, 선형 회귀 분석, 협업 필터링, 특성 값 학습, 주제 모델링에 사용
- 기울기 소멸 문제 해결 위해 사전 학습 용도로 활용 가능
- 심층 신뢰 신경망(DBN)의 요소로 활용됨

### 심층 신뢰 신경망(DBN)

입력층과 은닉층으로 구성된 제한된 볼츠만 머신(사전 훈련됨)을 블록처럼 여러 층으로 쌓은 형태로 연결된 신경망

- 레이블 없는 데이터에 대한 비지도 학습 가능  
->  부분적인 이미지에서 전체를 연상하는 일반화와 추상화 과정 구현에서 사용시 유용
- 순차적으로 심층 신뢰 신경망을 학습시켜 가면서 계층적 구조를 생성
- 위로 올라갈수록 추상적 특성을 추출
- 학습된 가중치를 디층 퍼셉트론의 가중치 초깃값으로 사용



학습 절차
1. 가시층과 은닉층1에 제한된 볼츠만 머신을 사전 훈련
2. 첫 번째 층 입력 데이터와 파라미터를 고정하여 두 번째 층 제한된 볼츠만 머신을 사전 훈련
3. 원하는 층 개수만큼 제한된 볼츠만 머신을 쌓아 올려 전체 DBN을 완성



## 4.4 우리는 무엇을 배워야 할까?

주어진 데이터를 활용해 어떤 결과 얻고 싶은지에 따라 머신러닝/딥러닝 선택 여부 나뉨.